%md
### TRANSFORMATION WORKFLOW

1. NORMALIZE COUNTRY (CNTRY)
2. CLEAN CUSTOMER_ID (remove hyphen)
3. RENAME COLUMNS
4. WRITE INTO SILVER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
# 0) LOAD DATA & READ FROM BRONZE
df_erploc = spark.table("acdproj.bronze.erp_loc_a101")

In [0]:
# 1) NORMALIZE CNTRY (country)
# Source mixes ISO codes, abbreviations, and full names for the same country
df_erploc = df_erploc.withColumn(
    "CNTRY",
    F.when(F.trim(F.col("CNTRY")).isin("US", "USA"), "United States")
     .when(F.trim(F.col("CNTRY")) == "DE", "Germany")
     .when(F.trim(F.col("CNTRY")).isNull() | (F.trim(F.col("CNTRY")) == ""), "n/a")
     .otherwise(F.trim(F.col("CNTRY")))
)

In [0]:
# 2) CLEAN CID (remove hyphen to align with crm_cust_info.customer_key format)
df_erploc = df_erploc.withColumn("CID", F.regexp_replace(F.col("CID"), "-", ""))

In [0]:
# 3 RENAME COLUMNS
RENAME_MAP_LOC = {
    "CID": "customer_id",
    "CNTRY": "country"
}
for old_name, new_name in RENAME_MAP_LOC.items():
    df_erploc = df_erploc.withColumnRenamed(old_name, new_name)

df_erploc.display()

In [0]:
# 4) WRITE INTO SILVER
df_erploc.write.mode("overwrite").saveAsTable("acdproj.silver.erp_locations")